# A3.3 · Egress control

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.2 · Sandboxed execution](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**.

| | |
|---|---|
| Tools used | Cilium, agentgateway |

## What this lesson is

**What it covers.** Attempt exfiltration to several destinations under an allow-list and see which survive.

**Why a security engineer needs it.** An agent with unrestricted egress turns any successful injection into data loss. The control it builds is: an allow-list at the network boundary, enforced where the agent cannot rewrite it.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Every exfiltration path in the architecture ends at the same place: a packet leaving your network. That makes egress the highest-leverage control you have, and the one most often left as allow-all because it broke something once.

> **At CyberTravels.** Every way customer PII leaves CyberTravels — a prompt leak, an abused tool, an OCR'd invoice, a poisoned template — ends at the same network boundary. R9, R10.

## 2 · The framework

```
   every exfiltration path, whatever its start, ends here:

   prompt leak  --+
   tool abuse   --+---> data assembled ---> [ egress ] ---> out
   code exec    --+                            ^
   memory read  --+                            |
                                    one place to enforce, one to log

   allow-all egress makes every control upstream best-effort
```

**Mitigates: T2 Tool Misuse · T6 Intent Breaking · LLM02 Sensitive Information Disclosure.**

Egress is the highest-leverage control in the architecture, for a structural
reason: **every exfiltration path ends at the network boundary**, no matter how
the agent was persuaded to take it.

Injection, tool misuse, a compromised MCP server, model-authored code, a
poisoned peer message — all of them converge on the same final step. Data leaves.
A control at that step does not need to understand what happened upstream, which
is exactly what makes it robust: it is the one place where you do not have to
predict the attack.

Two rules that decide whether it works:

**Allow-list, never deny-list.** You cannot enumerate the internet. A deny-list
blocks the destinations you thought of.

**Specific destinations.** `*.googleapis.com` or `*.s3.amazonaws.com` is not an
egress policy — anyone can create a bucket in those namespaces, and A1.3's
attacker will. The allow-list holds the destinations this workload actually
needs, and there are usually fewer than five.

And one placement rule: enforce it **where the agent cannot rewrite it** — the
network layer, the sidecar, the gateway. An egress check inside the agent's own
process is a suggestion to the component being attacked.

The cost is honest: an agent that needs the open internet cannot have this
control, and that is a design decision to make deliberately rather than by
default.

> **What this control closes.**
>
> The one control that does not need to know how the attack worked, because every exfiltration path ends here.

## 3 · Probing the sandbox, as a skill

An allow-list in a config file is a claim. CyberTravels' Coding Agent runs model-authored code, so the claim worth making is *this runtime has no egress*, and that one is settled by probing, in an environment you own. The procedure names the residual paths — DNS first, because it is the one that has actually been used — and requires you to record what you did **not** test, because a probe list is a statement about coverage. This is the file in this repository:

In [ ]:
# skills/attestation/sandbox-egress-verifier/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: sandbox-egress-verifier
description: >-
  Verify AI-generated code executes only in an approved sandbox with no
  network egress, and probe the known bypass paths. Use when attesting a no-
  egress execution claim, choosing a code-execution runtime, or asked
  whether a sandbox actually contains model-authored code.
allowed-tools: Bash, Read
---

# Sandbox Egress Verifier

**Controls:** Control 2 — sandboxed no-egress runtime

## Confidence: PARTIAL — and this ceiling is not negotiable

You can prove a sandbox is **misconfigured**. You cannot prove the negative
"no covert channel exists". Network isolation has documented bypass paths, and
a verdict of PASS on this control would be a claim the evidence cannot support.

**Cap every verdict this skill produces at PARTIAL.**

## Procedure

1. **Identify the runtime and its network mode.** Record which isolation
   technology is in use and which mode it runs in. A mode that permits general
   outbound network access is an immediate FAIL — no probing required.

2. **Confirm the runtime is on the approved list.** An approved sandbox in the
   wrong mode and an unapproved sandbox in the right mode are both findings.

3. **Probe the known residual paths**, in an environment you own:
   - **DNS.** Name resolution is frequently permitted where general egress is
     not, and A/AAAA queries carry data outward. This has been demonstrated to
     yield interactive command-and-control on a major managed sandbox.
   - **Object storage reachability.** Managed storage endpoints are commonly
     left reachable and have been used as a command-and-control channel.
   - **Host escape.** Subprocess spawning and any path from the sandboxed
     process to the host.

4. **Record what you did not test.** A probe list is a statement about coverage,
   and coverage is the honest part of this verdict.

## Output contract

```json
{
  "deployment_id": "str",
  "sandbox_runtime": "str",
  "network_mode": "str",
  "approved_runtime": true,
  "bypass_probes": [{"path": "dns|object_storage|host_escape", "reachable": false, "detail": "str"}],
  "untested_paths": ["str"],
  "verdict": "PARTIAL|FAIL",
  "verdict_ceiling_reason": "absence of a covert channel is not provable at runtime"
}
```

`PASS` is not a permitted value. If a tool emits one, that tool is wrong.

## Failure modes

- **Reporting PASS** because the configuration looked right. That is the whole
  reason this skill has a ceiling.
- **Probing only general HTTP egress.** DNS is the path that has actually been
  used.
- **Testing against someone else's infrastructure.** Probe only what you own.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/attestation/sandbox-egress-verifier/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/attestation/sandbox-egress-verifier/scripts/sandbox_egress_verifier.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Evaluate the same destinations under a deny-list and under an exact allow-list.

This is the executable half of the `sandbox-egress-verifier` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

ALLOW = {"api.corp.example", "reports-db.corp.example"}
DENY_SUFFIXES = {".evil.example"}          # the deny-list, for comparison

def by_denylist(host):
    return not any(host.endswith(s) for s in DENY_SUFFIXES)

def by_allowlist(host):
    return host in ALLOW                    # exact, not suffix

DESTINATIONS = [
 ("api.corp.example",              "the one it actually needs"),
 ("archive.evil.example",          "A1.3's exfiltration target"),
 ("attacker-bucket.s3.amazonaws.com", "a bucket anyone can create"),
 ("169.254.169.254",               "cloud metadata - every credential"),
 ("pastebin.example",              "not on anyone's deny-list"),
]

print(f"{'destination':38s}{'deny-list':12s}{'allow-list':12s}note")
for host, note in DESTINATIONS:
    d, a = by_denylist(host), by_allowlist(host)
    print(f"{host:38s}{'allow' if d else 'block':12s}{'allow' if a else 'block':12s}{note}")

leaked = [h for h, _ in DESTINATIONS if by_denylist(h) and h not in ALLOW]
print(f"\ndeny-list lets through : {len(leaked)}  {leaked}")
print(f"allow-list lets through : {sorted(h for h, _ in DESTINATIONS if by_allowlist(h))}")
print()
print("The deny-list blocked exactly the destination somebody had already")
print("thought of. It cannot be completed, because the internet cannot be")
print("enumerated.")
print()
print("Placement matters as much: this check belongs in the network path, not")
print("in the agent. A check inside the process being attacked is advice.")
assert len(leaked) == 3 and len([h for h, _ in DESTINATIONS if by_allowlist(h)]) == 1

## What you just proved

The skill loads and reports its shape. Its ceiling is PARTIAL and not negotiable: a configuration that looks right is not a PASS, and probing general HTTP while leaving DNS alone tests the path nobody uses. The untested list is part of the output, not an omission from it.

## Your turn

Write the allow-list for one agent by listing the hosts it genuinely calls. If it is under five, you can ship this control this week; if it is unbounded, that is the finding.

---

**Next → [A3.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*